# 演習10. 壊さないように測る

## シーン

ここまで、「一番遅い段は Infer です」と**答えを先に教えてもらって**進めてきました。
本番ではそうはいきません。**どの段が遅いのかは、測らないと分かりません。**

そして本番でやる順番は、いつもこれです。

```
[1] まず直列（1スレッド）で測る   … 1フレームの内訳を出す
[2] 内訳を見て、どこを直すか決める
[3] スレッドを組む
[4] もう一度測って、「やはりここだった」を確かめる
```

**なぜ [1] を直列でやるのか。** スレッドが動いている状態では、
「正味の処理時間」と「他のスレッドを待っていた時間」が混ざってしまうからです。
直列なら混ざりません。**1フレームの内訳が、そのまま出ます。**

そして、この演習でいちばん時間をかけるのはここです。

> **測ること自体が、測りたいものを変えてしまう。**

デバッグ表示を1行足しただけで結果が2倍変わることがあります。
そして**変わったことに気づけません**。数字はもっともらしく出続けるからです。

## 10-1. 【予測クイズ】計測は、いくらかかるのか

計測でやることは2種類しかありません。**時計を読むこと**と、**結果を出すこと**です。

```cpp
auto t = steady_clock::now();          // 時計を読む
...仕事...
busy += 経過時間;                       // 足し込む
cout << "time= " << ... << flush;      // 結果を出す
```

**実行する前に予測してください。** 1回あたり、それぞれ何ナノ秒くらいでしょうか。
そして、**どれが一番高い**でしょうか。

（1000万回まわすので、表示は `> /dev/null` に捨てます。
測定結果は `cerr` に出しているので画面に残ります）

In [ ]:
%%writefile ex10a.cpp
#include <iostream>
#include <iomanip>
#include <sstream>
#include <chrono>
using namespace std::chrono;

volatile long long sink = 0;

template <class F>
double ns_per_call(int n, F f) {          // 1回あたりのナノ秒
    auto t0 = steady_clock::now();
    for (int i = 0; i < n; i++) f(i);
    return duration_cast<nanoseconds>(steady_clock::now() - t0).count() / (double)n;
}

int main() {
    long long acc = 0;

    // (1) 時計を1回読む
    double c_clock = ns_per_call(1000000, [&](int) {
        sink += steady_clock::now().time_since_epoch().count() & 1;
    });

    // (2) 測って足し込む（時計2回 + 引き算 + 足し算）＝ 計測1回ぶん
    double c_lap = ns_per_call(1000000, [&](int) {
        auto a = steady_clock::now();
        auto b = steady_clock::now();
        acc += duration_cast<nanoseconds>(b - a).count();
    });

    // (3) 1行ぶんの文字列を作るだけ（画面には出さない）
    double c_str = ns_per_call(100000, [&](int i) {
        std::ostringstream os;
        os << "\nrunYOLO preprocessing time= " << i << " [mS]\n";
        sink += os.str().size();
    });

    // (4) cout に1行流す（flush なし）
    double c_out = ns_per_call(20000, [&](int i) {
        std::cout << "\nrunYOLO preprocessing time= " << i << " [mS]\n";
    });

    // (5) cout に1行流して flush する
    double c_flush = ns_per_call(20000, [&](int i) {
        std::cout << "\nrunYOLO preprocessing time= " << i << " [mS]\n" << std::flush;
    });

    std::cerr << std::fixed << std::setprecision(0);
    std::cerr << "\n1回あたりの値段（このマシン、この出力先で）\n\n";
    std::cerr << std::setw(10) << c_clock << " ns   steady_clock::now() を1回読む\n";
    std::cerr << std::setw(10) << c_lap   << " ns   時計2回 + 引き算 + 足し込み（計測1回ぶん）\n";
    std::cerr << std::setw(10) << c_str   << " ns   1行ぶんの文字列を作る（表示はしない）\n";
    std::cerr << std::setw(10) << c_out   << " ns   cout に1行流す（flush なし）\n";
    std::cerr << std::setw(10) << c_flush << " ns   cout に1行流して flush する\n";
    std::cerr << "\n(" << acc << " " << sink << ")\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 ex10a.cpp -o ex10a && ./ex10a > /dev/null

### 10-1-1. 結果 ―― 時計は安い。表示は高い

```
        22 ns   steady_clock::now() を1回読む
        43 ns   時計2回 + 引き算 + 足し込み（計測1回ぶん）
       149 ns   1行ぶんの文字列を作る（表示はしない）
        49 ns   cout に1行流す（flush なし）
       163 ns   cout に1行流して flush する
```

**時計を読むのは 22ns。** 1フレームに 14 回読んでも 0.3µs です。
1フレームが 10ms なら、影響は **0.003%**。**気にする必要はありません。**

一方、**文字列を1行作るだけで 149ns**。時計7回ぶんです。
`ostringstream` を作り、数字を10進に変換し、メモリを確保するからです。

> **測るのは安い。出すのが高い。**

### 10-1-2. しかも「高さ」は出力先で3桁変わる

上の 163ns は、**出力先が `/dev/null` だから**の値です。実際にはこうなります。

```
ファイル / dev/null       0.2 〜 0.3 us
パイプ（| tee など）       0.8 us
SSH 端末                  数百 us
シリアルコンソール          約 3 ms      ← 1行で
```

シリアルの値は計算で出せます。**115200 baud は毎秒 11520 バイト**なので、
1バイト 87µs。36 バイトの行で **3.1ms** です。

段ごとの時間をその場で表示すると、**1フレームに4行**になります。

```
4行 x 3.1ms = 12.4 ms/フレーム
```

yolov3（1フレーム 82ms）なら 15%。**tiny（13ms 前後）なら処理時間と同じかそれ以上**です。

> **同じ `cout` が、Colab では無害で、シリアルコンソールでは処理より重い。**

### 10-1-3. いちばん怖いのは「高いのに、数字に出ない」こと

しかも、`cout` を**計測区間の外**に置いてあったとします。これ自体は正しい書き方です。

```cpp
auto t0 = now();
...前処理...
auto t1 = now();
cout << "pre time= " << (t1-t0) << flush;   // 計測区間の外
auto t2 = now();                            // dpu の計測はここから
```

つまり **`pre` も `dpu` も `post` も、正しい値を出し続けます。**
それなのに全体は 12ms/フレーム遅くなっています。

**気づく方法は1つだけです。**

> **「各段の合計」と「1フレームの実測時間」を並べて出し、差を見る。**

この差を、以降 **「計測外」** と呼びます。
ここが大きければ、**測り漏らしている仕事があります。**

## 10-2. 【予測クイズ】表示を鍵の中に入れると

もう1つ、よくある事故です。

```cpp
{
    std::lock_guard<std::mutex> g(mtx);
    long r = work(j);                          // 計算まで鍵の中に入っている
    std::cout << "job " << j << " -> " << r << "\n";
}
```

「表示が混ざらないように鍵をかけた」つもりが、**計算まで直列化しています。**
演習3で見た「鍵の区間が広すぎる」そのものです。

次のプログラムは、同じ計算を4通りの表示のしかたで実行します。

- 何もしない
- 1件ごとに表示（**鍵の外**）
- 1件ごとに表示（**鍵の中**）
- 結果を**貯めておいて、最後にまとめて出す**

**実行する前に予測してください。** どれが速く、どれが遅いでしょうか。

In [ ]:
%%writefile ex10b.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <mutex>
#include <chrono>
using namespace std::chrono;

const int NWORKER = 2;
const int NJOB    = 400;
std::mutex mtx;
long sink = 0;                           // 最適化で消されないように、結果を足しておく場所

long work(int j) {                       // 1件ぶんの計算（0.2msくらい）
    long s = 0;
    for (int i = 0; i < 200000; i++) s += (j + i) % 7;
    return s;
}

enum How { NONE, PRINT_OUTSIDE, PRINT_INSIDE, COLLECT };

int run(How how) {
    std::vector<std::vector<long>> log(NWORKER);     // COLLECT 用の記録場所
    auto t0 = steady_clock::now();

    std::vector<std::thread> ts;
    for (int k = 0; k < NWORKER; k++)
        ts.emplace_back([&, k] {
            for (int j = k; j < NJOB; j += NWORKER) {
                if (how == PRINT_INSIDE) {
                    std::lock_guard<std::mutex> g(mtx);          // 表示のために鍵をかける
                    long r = work(j);                            // 計算まで鍵の中に入っている
                    std::cout << "job " << j << " -> " << r << "\n";
                } else {
                    long r = work(j);                            // 計算は鍵の外
                    if (how == PRINT_OUTSIDE) std::cout << "job " << j << " -> " << r << "\n";
                    if (how == COLLECT)       log[k].push_back(r);   // 貯めるだけ
                }
            }
        });
    for (auto& t : ts) t.join();
    int ms = duration_cast<milliseconds>(steady_clock::now() - t0).count();

    for (auto& v : log) for (long r : v) sink += r;   // 貯めた分は最後にまとめて使う
    return ms;
}

int main() {
    // 表示の中身を見たいわけではないので、実行時に > /dev/null で捨てます。
    // 測定結果だけは std::cerr に出すので、画面に残ります。
    std::cerr << "計算 400件を2スレッドで。表示のしかただけを変える。\n\n";
    std::cerr << "  何もしない               : " << run(NONE)          << " ms\n";
    std::cerr << "  1件ごとに表示（鍵の外）  : " << run(PRINT_OUTSIDE) << " ms\n";
    std::cerr << "  1件ごとに表示（鍵の中）  : " << run(PRINT_INSIDE)  << " ms\n";
    std::cerr << "  貯めて最後にまとめて出す : " << run(COLLECT)       << " ms\n";
    std::cerr << "\n(合計 " << sink << ")\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex10b.cpp -o ex10b && ./ex10b > /dev/null

### 10-2-1. 結果 ―― 鍵の中で表示すると2倍遅い

だいたいこうなります（絶対値はマシンによって変わります。**比を見てください**）。

```
  何もしない               :  93 ms
  1件ごとに表示（鍵の外）  :  95 ms
  1件ごとに表示（鍵の中）  : 190 ms     ← 2倍
  貯めて最後にまとめて出す :  95 ms
```

**鍵の中で表示すると、ちょうど2倍。** スレッドが2本なので、
「2本立てているのに1本ぶんしか動いていない」状態です。

計測として最悪なのは、**この状態で測った数字を信じてしまう**ことです。

- 「並列にしたのに速くならない」 ⇒ 実は、デバッグ表示のせいで直列化していた
- 表示を消したら速くなった ⇒ **どこが本当のボトルネックだったのか、分からずじまい**

### 10-2-2. 壊さないように測る ―― 5つの作法

**① 実行中は足し込むだけ。表示は全部終わってから1回だけ**

上の結果でも、貯める版は何もしない版と同じ速さでした。

**② 各スレッドが自分専用の記録場所を持つ**

共有しないので、**鍵が要りません。** 鍵が要らないので、直列化も起きません。

**③ `steady_clock` と `microseconds`**

`system_clock` は時刻同期で飛びます。
そして **`milliseconds` は切り捨て**なので、`2.9ms` が `2` になります。
DPU が 2ms の tiny では、これだけで数十パーセントずれます。

**④ 立ち上がりの数フレームを捨てる**

最初の数枚はメモリ確保やキャッシュの空振りを含みます。

**⑤ 平均と最大の両方を出し、「合計 = 実測」を検算する**

平均だけでは「たまに 300ms」という段が見えません。
そして 10-1-3 のとおり、**合計と実測の差＝計測外**が測り漏らしを教えてくれます。

### 10-2-3. 道具の形

この5つを満たす道具は、驚くほど小さく書けます。

```cpp
void lap(int id, TP& t) {                   // t から今までを段 id に足す
    TP now = steady_clock::now();
    long long us = duration_cast<microseconds>(now - t).count();
    t = now;                                // 物差しを進める
    if (warm_) return;                      // ④ 立ち上がりは捨てる
    st_[id].sum += us;                      // ① 足し込むだけ
    if (us > st_[id].max) st_[id].max = us; // ⑤ 最大も
    frame_ += us;                           // ⑤ 検算用の合計
}
```

**`t` を参照で受け取って進めていく**のが要点です。
こうすると段と段のあいだに隙間ができないので、**測り漏らしがそのまま「計測外」に出ます。**

次で、この道具を実際に使います。

## 10-3. 【予測クイズ】直列で1フレームの内訳を測る

10-2-3 の道具を、実際に使います。仕事の中身は **本番の tiny を模したもの**です。

```
              計算    待ち
Read          13ms      -     デコード
pre            9ms      -     前処理（resize 4ms / quantize 5ms）
dpu             -      2ms    DPU に投げて待つ
post            2ms     -     後処理
show             -     30ms   imshow（10枚に1枚は 60ms）
```

**まだスレッドは組みません。** 直列のまま、1フレームの内訳を出すだけです。

**実行する前に予測してください。**

- 内訳で**一番大きい段**はどれでしょうか
- その段は「計算」でしょうか、「待ち」でしょうか
- **スレッドを組めば速くなる**でしょうか。それとも、組んでも無駄でしょうか

In [ ]:
%%writefile ex10c.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <vector>
#include <chrono>
using namespace std::chrono;

using TP = steady_clock::time_point;
static inline TP now_() { return steady_clock::now(); }

// ============ 計測の道具 ============ 実行中は足し込むだけ。表示は最後に1回だけ
class Profiler {
public:
    // wait = CPU を使わずに待つだけの段 / child = ひとつ上の段の内訳（合計に足さない）
    int stage(const char* name, bool wait, bool child = false) {
        st_.push_back({name, wait, child, 0, 0});
        return (int)st_.size() - 1;
    }
    void lap(int id, TP& t) {                       // t から今までを段 id に足し、t を進める
        TP n = now_();
        long long us = duration_cast<microseconds>(n - t).count();
        t = n;
        if (warm_) return;                          // 立ち上がりは捨てる
        st_[id].sum += us;
        if (us > st_[id].max) st_[id].max = us;
        if (!st_[id].child) frame_ += us;           // 検算用の合計
    }
    void frame_end(TP fstart) {
        long long total = duration_cast<microseconds>(now_() - fstart).count();
        if (warm_) { if (++skipped_ >= WARMUP) { warm_ = false; t0_ = now_(); } return; }
        other_ += (total > frame_ ? total - frame_ : 0);   // 測り漏らし ＝ 計測外
        frame_ = 0;
        frames_++;
    }
    void report(double cores) const {
        double sec = duration_cast<microseconds>(now_() - t0_).count() / 1e6;
        double per = 1000.0 * sec / frames_;
        std::cout << std::fixed << std::setprecision(1)
                  << "有効 " << frames_ << " 枚（先頭 " << WARMUP << " 枚は捨てた）  "
                  << (frames_ / sec) << " FPS   1フレーム平均 " << per << "ms\n\n"
                  << "     平均      最大    割合  種類  段\n";
        double slow = 0, cpu = 0;
        const char* slow_name = "";
        for (const auto& s : st_) {
            double avg = s.sum / 1000.0 / frames_, mx = s.max / 1000.0;
            std::cout << std::setw(9) << avg << "ms" << std::setw(8) << mx << "ms";
            if (s.child) { std::cout << "               - " << s.name << "\n"; continue; }
            std::cout << std::setw(7) << (100.0 * avg / per) << "%"
                      << (s.wait ? "  待ち  " : "  計算  ") << s.name << "\n";
            if (avg > slow) { slow = avg; slow_name = s.name; }
            if (!s.wait) cpu += avg;
        }
        double oth = other_ / 1000.0 / frames_;
        std::cout << std::setw(9) << oth << "ms" << std::setw(10) << "-"
                  << std::setw(7) << (100.0 * oth / per) << "%          計測外\n";

        double A = 1000.0 / slow, B = 1000.0 * cores / cpu;
        std::cout << "\n上限A = 1000 / " << slow << "ms（一番遅い段 " << slow_name << "） = "
                  << A << " FPS\n"
                  << "上限B = 1000 x " << std::setprecision(2) << cores << std::setprecision(1)
                  << " / " << cpu << "ms（「待ち」を除いた合計） = " << B << " FPS\n"
                  << (A < B ? std::string("=> 効いているのは 上限A。段を分ければ ") + slow_name + " を隠せるかもしれません\n"
                            : std::string("=> 効いているのは 上限B。スレッドを増やしても伸びません\n"));
    }
private:
    struct S { const char* name; bool wait, child; long long sum, max; };
    static const int WARMUP = 3;
    std::vector<S> st_;
    long long frame_ = 0, other_ = 0;
    int frames_ = 0, skipped_ = 0;
    bool warm_ = true;
    TP t0_ = now_();
};

// ============ 仕事の中身（本番の tiny を模したもの） ============
long calib = 0;
volatile long sink = 0;
long burn(long n) { long s = 0; for (long i = 0; i < n; i++) s += (i * 2654435761u) % 7; return s; }
void cpu_ms(int ms)  { sink += burn(calib * ms); }
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }

void calibrate() {
    long n = 100000;
    for (;;) {
        TP t = now_(); sink += burn(n);
        auto us = duration_cast<microseconds>(now_() - t).count();
        if (us > 30000) { calib = n * 1000 / us; break; }
        n *= 2;
    }
}
// hardware_concurrency() は当てにならない（演習7-2-4）。同時に進む計算の量を実測する
double effective_cores(unsigned hw) {
    long U = calib * 60;
    TP t = now_(); sink += burn(U);
    double one = duration_cast<microseconds>(now_() - t).count();
    std::vector<std::thread> th;
    t = now_();
    for (unsigned k = 0; k < hw; k++) th.emplace_back([&] { sink += burn(U); });
    for (auto& x : th) x.join();
    double many = duration_cast<microseconds>(now_() - t).count();
    return hw * one / many;
}

int main() {
    calibrate();
    unsigned hw = std::thread::hardware_concurrency();
    double eff = effective_cores(hw);
    std::cout << std::fixed << std::setprecision(2)
              << "hardware_concurrency() = " << hw
              << " ですが、同時に進む計算の量を実測すると " << eff << " 個分でした\n\n";

    Profiler p;
    int READ = p.stage("read (デコード)", false);
    int PRE  = p.stage("pre  (前処理)",   false);
    int RS   = p.stage("resize",          false, true);   // pre の内訳
    int QT   = p.stage("quantize",        false, true);   // pre の内訳
    int DPU  = p.stage("dpu  (推論)",     true);          // 投げて待つだけ
    int POST = p.stage("post (後処理)",   false);
    int SHOW = p.stage("show (表示)",     true);

    for (int i = 0; i < 33; i++) {
        TP t = now_(), fstart = t;

        cpu_ms(13);                                p.lap(READ, t);   // デコード

        TP ts = t;                                                   // pre の内訳用
        cpu_ms(4);                                 p.lap(RS, ts);
        cpu_ms(5);                                 p.lap(QT, ts);
                                                   p.lap(PRE, t);

        wait_ms(2);                                p.lap(DPU, t);    // DPU に投げて待つ
        cpu_ms(2);                                 p.lap(POST, t);   // 後処理
        wait_ms(i % 10 == 0 ? 60 : 30);            p.lap(SHOW, t);   // imshow

        p.frame_end(fstart);
    }
    p.report(eff);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread -O2 ex10c.cpp -o ex10c && ./ex10c

## 10-4. 結果の読み方

```
hardware_concurrency() = 2 ですが、同時に進む計算の量を実測すると 1.66 個分でした

有効 30 枚（先頭 3 枚は捨てた）  16.7 FPS   1フレーム平均 59.8ms

     平均      最大    割合  種類  段
     13.4ms    15.9ms   22.4%  計算  read (デコード)
      9.1ms    10.9ms   15.3%  計算  pre  (前処理)
      4.1ms     4.7ms               - resize
      5.1ms     6.2ms               - quantize
      2.1ms     2.1ms    3.5%  待ち  dpu  (推論)
      2.0ms     2.4ms    3.4%  計算  post (後処理)
     33.1ms    60.1ms   55.4%  待ち  show (表示)
      0.0ms         -    0.0%          計測外

上限A = 1000 / 33.1ms（一番遅い段 show (表示)） = 30.2 FPS
上限B = 1000 x 1.66 / 24.6ms（「待ち」を除いた合計） = 67.6 FPS
=> 効いているのは 上限A。段を分ければ show (表示) を隠せるかもしれません
```

読む順番は3つです。

### 10-4-1. まず「計測外」を見る

`0.0ms`。**測り漏らしはありません。**

ここが大きければ、**他のどの数字も信用してはいけません。**
段と段のあいだに、測っていない仕事が挟まっているということです。
`cout` かもしれませんし、デコードのように「測るのを忘れていた仕事」かもしれません。
**まずそれを潰してから、先に進みます。**

### 10-4-2. 一番大きい段と、その「種類」を見る

`show 33.1ms`、種類は **待ち**。ここが決定的です。

- **「計算」の段が一番大きい** ⇒ その計算そのものを減らすしかない
- **「待ち」の段が一番大きい** ⇒ **段を分ければ、他の段の裏に隠せる**

いまは後者なので、**「スレッドを組めば効く」と予測できます。**
しかも上限A（30.2）が上限B（67.6）よりずっと小さいので、
**まだコアには余裕があります。**

逆に上限B のほうが小さければ、**スレッドを組んでも無駄**でした（演習7-2-3）。
**ここまでを直列のまま判断できる**のが、直列で測る理由です。

### 10-4-3. 「最大」の列を見る

`show` は平均 33ms なのに**最大 60ms**。2倍近く振れています。
本番の `imshow` はもっとひどく、平均の10倍まで跳ねます。

**平均だけ見ていたら、この振れに気づけません。**
そして振れる段は、パイプラインにしたときにキューを詰まらせる原因になります。

（`read` の最大が平均より大きいのは、Colab が共有マシンだからです）

### 10-4-4. 手順のまとめ

```
[1] 直列で測る
      ・計測外はゼロか
      ・一番大きい段はどれか。それは「計算」か「待ち」か
      ・上限A と 上限B のどちらが効いているか
[2] 決める
      待ちが大きい  -> 段を分けて隠す
      計算が大きい  -> その計算を減らす
      上限B が効く  -> スレッドではなく計算量
[3] 組む
[4] 段ごとに 正味 / 取り出し待ち / 入れ待ち を測って確かめる
      ・待っていない段が、[1] で予測した段か
      ・実測が、[1] から計算した上限A に近いか
      -> 違っていたら、[1] の測り方を疑う
```

**[3] と [4] は発展課題でやります。**
そして直したら [1] に戻ります。**ボトルネックは移動しているからです。**

## 10-5. 測り方の落とし穴

**① 1回だけ測って判断しない**

同じプログラムでも実行のたびに 10% は動きます。
5%の差を1回の測定で語ってはいけません。
**数回まわして、最小値と中央値を見てください。1回目は捨てます。**

**② 途中で打ち切らない**

打ち切ると、**キューに残っている仕事を数え忘れます**。

**③ 「速くなった」の中身を言えるようにする**

FPS が上がったのか、レイテンシが縮んだのか。
演習7で見たとおり、**この2つは同時には良くなりません。**

**④ 粗いところから測る**

```
1. 全体の時間（FPS）を測る                 <- まずここ
2. 直列で1フレームの内訳を測る              <- 犯人の段が分かる
3. 犯人の段の中を分けて測る                 <- resize なのか quantize なのか
```

1と2で答えが出ることがほとんどです。

**⑤ 計測環境を変えたら、前の数字は捨てる**

コンパイルオプション、入力、表示の有無 ―― どれか1つでも変えたら、
比較できるのは**同じ条件で測り直した数字だけ**です。

## 発展課題

10-4-4 の [3] と [4] です。**測ってから組み、組んでから確かめてください。**

1. この仕事を **3段パイプライン**（Read / Infer / Show）にしたら、FPS はいくつになるでしょうか。
   **まず 10-4 の表から予測**してください。そのうえで、段ごとに
   `正味` / `取り出し待ち` / `入れ待ち` を測り、**待っていない段**を探してください。
   それは予測した段でしょうか。

2. 犯人が分かったとして、**その段を軽くしたら**どうなるでしょうか。
   本番でいえば `imshow` をやめて、圧縮して PC に送って表示するような変更です。
   **Show を 30ms から 4ms にしたとき**、FPS はどこまで伸びますか。
   そして、**次の犯人はどの段**になるでしょうか。

---

## 演習の地図 ―― ここまでで扱ったこと

```
1  スレッドとパイプライン    thread / join、待つ仕事と計算する仕事、レイテンシとスループット
2  データ競合                共有データが壊れる、atomic で足りる場合と足りない場合
3  mutex で守る              lock_guard / RAII、鍵の区間の広さ、デッドロック
4  待ち合わせ                ポーリングの無駄、condition_variable、述語
5  スレッドセーフなキュー    鍵1本＋条件変数2本、容量つきキューを組み立てる
6  容量とバックプレッシャ    容量で変わるもの・変わらないもの、遅れとメモリ
7  パイプラインを組む        2つの上限、ボトルネックの移動、どう速くしていくか
8  順序が崩れる              並列化の副作用、番号を持たせる、出口で並べ直す
9  安全な終了                終了を下流へ伝える、close / 番兵、join が返る形
10 壊さないように測る        測る値段、計測外、直列で測って組んで確かめる
```

本番では、4スレッド + キュー2本のパイプラインを読み、改造します。
読むときは、まずこの3つを確かめてください。

1. **段はいくつあって、どのキューでつながっているか**
2. **鍵は何本で、条件変数は何本で、それぞれ何を待っているか**
3. **終了は、どうやって下流に伝わっているか**

そして改造するときは、**必ず先に、直列で測ってください。**